In [ ]:
"""Workbook to create figures destined for the flagship paper."""

# pylint: disable=import-error, redefined-outer-name, use-dict-literal, too-many-lines, too-many-branches, duplicate-code

## SETUP

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from __future__ import annotations

import copy
import json  # pylint: disable=unused-import
import re
import tarfile
from pathlib import Path
from typing import IO, Any, Dict, List, Tuple

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "notebook"
from IPython.display import display  # pylint: disable=unused-import
from plotly.subplots import make_subplots
from scipy import stats

from epiclass.utils.bed_utils import bed_to_bins
from epiclass.utils.notebooks.paper.paper_utilities import (
    ASSAY,
    ASSAY_MERGE_DICT,
    ASSAY_ORDER,
    CELL_TYPE,
    IHECColorMap,
    MetadataHandler,
    SplitResultsHandler,
    extract_input_sizes_from_output_files,
    save_figure,
)

In [ ]:
base_dir = Path.home() / "Projects/epiclass/output/paper"
paper_dir = base_dir
if not paper_dir.exists():
    raise FileNotFoundError(f"Directory {paper_dir} does not exist.")

base_data_dir = base_dir / "data"
base_fig_dir = base_dir / "figures"

In [ ]:
chromsize_path = base_data_dir / "chromsizes" / "hg38.noy.chrom.sizes"
if not chromsize_path.exists():
    raise FileNotFoundError(f"File {chromsize_path} does not exist.")

with open(chromsize_path, "r", encoding="utf-8") as f:
    pairs = [line.rstrip("\n").split() for line in f]

chromsizes: List[Tuple[str, int]] = sorted([(name, int(size)) for name, size in pairs])
chroms: List[str] = [name for name, _ in chromsizes]

In [ ]:
IHECColorMap = IHECColorMap(base_fig_dir)
assay_colors = IHECColorMap.assay_color_map
cell_type_colors = IHECColorMap.cell_type_color_map

In [ ]:
split_results_handler = SplitResultsHandler()

metadata_handler = MetadataHandler(paper_dir)
metadata = metadata_handler.load_metadata("v2")
metadata_df = metadata.to_df()

In [ ]:
gen_data_dir = base_data_dir / "training_results" / "dfreeze_v2"
if not gen_data_dir.exists():
    raise FileNotFoundError(f"Directory {gen_data_dir} does not exist.")

data_dir_100kb = gen_data_dir / "hg38_100kb_all_none"
if not data_dir_100kb.exists():
    raise FileNotFoundError(f"Directory {data_dir_100kb} does not exist.")

In [ ]:
# save pre-plot data
plot_data_dir = base_fig_dir / "flagship" / "pre-plot"
plot_data_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
fig_N = "4"

## Panel D - Cell type accuracy per assay, for different training regions

In [ ]:
mixed_data_dir = gen_data_dir / "mixed"
if not mixed_data_dir.exists():
    raise FileNotFoundError(f"Directory {mixed_data_dir} does not exist.")

In [ ]:
flagship_selection_4cat = [
    "hg38_cpg_topvar_200bp_n303k_wrong_coordinates",
    "hg38_regulReg_allCorr_n303k",
    "hg38_gene_regions_100kb_coord_n19864",
    "hg38_100kb_all_none",
]

metric_orders_map = {
    "flagship_selection_4cat": flagship_selection_4cat,
    # "fig1_sets": fig1_sets,
    # "feature_sets_14": feature_sets_14,
    # "different_nature_sets": different_nature_sets,
    # "feature_set_compare_cpg_regul": feature_set_compare_cpg_regul,
}

In [ ]:
set_selection_name = "flagship_selection_4cat"

In [ ]:
input_sizes = extract_input_sizes_from_output_files(mixed_data_dir)  # type: ignore
input_sizes: Dict[str, int] = {k: v.pop() for k, v in input_sizes.items() if len(v) == 1}  # type: ignore

In [ ]:
all_metrics = split_results_handler.obtain_all_feature_set_data(
    parent_folder=mixed_data_dir,
    merge_assays=True,
    return_type="metrics",
    include_categories=[CELL_TYPE],
    include_sets=metric_orders_map[set_selection_name],
    exclude_names=["16ct", "27ct", "7c", "chip-seq-only"],
)


# Order the metrics
all_metrics = {
    name: all_metrics[name]  # type: ignore
    for name in metric_orders_map[set_selection_name]
    if name in all_metrics
}

In [ ]:
# correct a name
try:
    all_metrics["hg38_100kb_all_none"][ASSAY] = all_metrics["hg38_100kb_all_none"][  # type: ignore
        f"{ASSAY}_11c"
    ]
    del all_metrics["hg38_100kb_all_none"][f"{ASSAY}_11c"]
except KeyError:
    pass

In [ ]:
resolution_colors = {
    "100kb": px.colors.qualitative.Safe[0],
    "10kb": px.colors.qualitative.Safe[1],
    "1kb": px.colors.qualitative.Safe[2],
    "regulReg": px.colors.qualitative.Safe[3],
    "gene": px.colors.qualitative.Safe[4],
    "cpg": px.colors.qualitative.Safe[5],
    "1mb": px.colors.qualitative.Safe[6],
    "5mb": px.colors.qualitative.Safe[7],
    "10mb": px.colors.qualitative.Safe[8],
}

#### Global Average

In [ ]:
def graph_feature_set_metrics(
    all_metrics: Dict[str, Dict[str, Dict[str, Dict[str, float]]]],
    input_sizes: Dict[str, int],
    logdir: Path | str | None = None,
    sort_by_input_size: bool = False,
    name: str | None = None,
    y_range: Tuple[float, float] | None = None,
    boxpoints: str = "all",
    width: int = 1200,
    height: int = 1200,
    verbose: bool = False,
) -> None:
    """Graph the metrics for all feature sets.

    Args:
        all_metrics (Dict[str, Dict[str, Dict[str, Dict[str, float]]]): A dictionary containing all metrics for all feature sets.
            Format: {feature_set: {task_name: {split_name: metric_dict}}}
        input_sizes (Dict[str, int]): A dictionary containing the input sizes for all feature sets.
        logdir (Path): The directory where the figure will be saved. If None, the figure will only be displayed.
        sort_by_input_size (bool): Whether to sort the feature sets by input size.
        name (str|None): The name of the figure.
        y_range (Tuple[float, float]|None): The y-axis range for the figure.
        boxpoints (str): The type of boxpoints to display. Can be "all" or "outliers". Defaults to "all".
    """
    if boxpoints not in ["all", "outliers"]:
        raise ValueError("Invalid boxpoints value.")

    reference_hdf5_type = "hg38_100kb_all_none"
    metadata_categories = list(all_metrics[reference_hdf5_type].keys())

    used_resolutions = set()
    for i in range(len(metadata_categories)):
        category_idx = i
        category_fig = make_subplots(
            rows=1,
            cols=2,
            shared_yaxes=True,
            subplot_titles=["Accuracy", "F1-score (macro)"],
            # x_title="Feature set",
            y_title="Metric value",
        )

        trace_names = []
        order = list(all_metrics.keys())
        if sort_by_input_size:
            order = sorted(
                all_metrics.keys(),
                key=lambda x: input_sizes[x],
            )
        for feature_set_name in order:
            if verbose:
                print(f"Processing feature set: {feature_set_name}")

            tasks_dicts = all_metrics[feature_set_name]
            meta_categories = copy.deepcopy(metadata_categories)

            if feature_set_name not in input_sizes:
                print(f"Skipping {feature_set_name}, no input size found.")
                continue

            task_name = meta_categories[category_idx]
            if "split" in task_name:
                raise ValueError("Split in task name. Wrong metrics dict.")

            try:
                task_dict = tasks_dicts[task_name]
            except KeyError as err:
                if verbose:
                    print(f"KeyError for {feature_set_name}, {task_name}: {err}")
                else:
                    print("Skipping", feature_set_name, task_name)
                    continue

            input_size = input_sizes[feature_set_name]

            feature_set_name = feature_set_name.replace("_none", "")
            feature_set_name = feature_set_name.replace("hg38_", "")

            resolution = feature_set_name.split("_")[0]
            used_resolutions.add(resolution)

            trace_name = f"{input_size}|{feature_set_name}"
            trace_names.append(trace_name)

            # Accuracy
            metric = "Accuracy"
            y_vals = [task_dict[split][metric] for split in task_dict]
            hovertext = [
                f"{split}: {metrics_dict[metric]:.4f}"
                for split, metrics_dict in task_dict.items()
            ]
            category_fig.add_trace(
                go.Box(
                    y=y_vals,
                    name=trace_name,
                    boxmean=True,
                    boxpoints=boxpoints,
                    marker=dict(size=3, color="black"),
                    line=dict(width=1, color="black"),
                    fillcolor=resolution_colors[resolution],
                    hovertemplate="%{text}",
                    text=hovertext,
                    legendgroup=resolution,
                    showlegend=False,
                ),
                row=1,
                col=1,
            )

            metric = "F1_macro"
            y_vals = [task_dict[split][metric] for split in task_dict]
            hovertext = [
                f"{split}: {metrics_dict[metric]:.4f}"
                for split, metrics_dict in task_dict.items()
            ]
            category_fig.add_trace(
                go.Box(
                    y=y_vals,
                    name=trace_name,
                    boxmean=True,
                    boxpoints=boxpoints,
                    marker=dict(size=3, color="black"),
                    line=dict(width=1, color="black"),
                    fillcolor=resolution_colors[resolution],
                    hovertemplate="%{text}",
                    text=hovertext,
                    legendgroup=resolution,
                    showlegend=False,
                ),
                row=1,
                col=2,
            )

        title = f"Neural network performance - {metadata_categories[category_idx]}"
        if name is not None:
            title += f" - {name}"
        category_fig.update_layout(
            width=width,
            height=height,
            title=title,
        )

        # dummy scatters for resolution colors
        for resolution, color in resolution_colors.items():
            if resolution not in used_resolutions:
                continue
            category_fig.add_trace(
                go.Scatter(
                    x=[None],
                    y=[None],
                    mode="markers",
                    name=resolution,
                    marker=dict(color=color, size=5),
                    showlegend=True,
                    legendgroup=resolution,
                )
            )

        category_fig.update_layout(legend=dict(itemsizing="constant"))

        # y-axis
        if y_range:
            category_fig.update_yaxes(range=y_range)
        else:
            if ASSAY in task_name:
                category_fig.update_yaxes(range=[0.96, 1.001])
            if CELL_TYPE in task_name:
                category_fig.update_yaxes(range=[0.75, 1])

        # Save figure
        if logdir:
            logdir = Path(logdir)
            base_name = f"feature_set_metrics_{metadata_categories[category_idx]}"
            if name is not None:
                base_name = base_name + f"_{name}"
            save_figure(category_fig, logdir, base_name)

        category_fig.show()

In [ ]:
# preplot_path = plot_data_dir / f"fig{fig_N}_D_metrics.json"
# with open(preplot_path, "w", encoding="utf-8") as f:
#     json.dump(all_metrics, f, indent=4)

# preplot_path = plot_data_dir / f"fig{fig_N}_D_input_sizes.json"
# with open(preplot_path, "w", encoding="utf-8") as f:
#     json.dump(input_sizes, f, indent=4)

In [ ]:
graph_feature_set_metrics(
    all_metrics=all_metrics,  # type: ignore
    input_sizes=input_sizes,
    boxpoints="all",
    width=700,
    height=800,
    y_range=(0.295, 1.005),
)

#### Metrics per assay

In [ ]:
def prepare_metric_sets_per_assay(
    all_results: Dict[str, Dict[str, Dict[str, pd.DataFrame]]], verbose: bool = False
) -> Dict[str, Dict[str, Dict[str, Dict[str, Dict[str, float]]]]]:
    """Prepare metric sets per assay.

    Args:
        all_results (Dict[str, Dict[str, Dict[str, pd.DataFrame]]]): A dictionary containing all results for all feature sets.

    Returns:
        Dict[str, Dict[str, Dict[str, Dict[str, float]]]]: A dictionary containing all metrics per assay for all feature sets.
            Format: {assay: {feature_set: {task_name: {split_name: metric_dict}}}}
    """
    if verbose:
        print("Loading metadata.")
    metadata = metadata_handler.load_metadata("v2")
    metadata.convert_classes(ASSAY, ASSAY_MERGE_DICT)
    md5_per_assay = metadata.md5_per_class(ASSAY)
    md5_per_assay = {k: set(v) for k, v in md5_per_assay.items()}

    if verbose:
        print("Getting results per assay.")
    results_per_assay = {}
    for assay_label in ASSAY_ORDER:
        if verbose:
            print(assay_label)
        results_per_assay[assay_label] = {}
        for feature_set, task_dict in all_results.items():
            if verbose:
                print(feature_set)
            results_per_assay[assay_label][feature_set] = {}
            for task_name, split_dict in task_dict.items():
                if verbose:
                    print(task_name)
                results_per_assay[assay_label][feature_set][task_name] = {}

                # Only keep the relevant assay
                for split_name, split_df in split_dict.items():
                    if verbose:
                        print(split_name)
                    assay_df = split_df[split_df.index.isin(md5_per_assay[assay_label])]
                    results_per_assay[assay_label][feature_set][task_name][
                        split_name
                    ] = assay_df

    if verbose:
        print("Finished getting results per assay. Now computing metrics.")
    metrics_per_assay = {}
    for assay_label in ASSAY_ORDER:
        if verbose:
            print(assay_label)
        metrics_per_assay[assay_label] = {}
        for feature_set, task_dict in results_per_assay[assay_label].items():
            if verbose:
                print(feature_set)
            assay_metrics = split_results_handler.compute_split_metrics(
                task_dict, concat_first_level=True
            )
            inverted_dict = split_results_handler.invert_metrics_dict(assay_metrics)
            metrics_per_assay[assay_label][feature_set] = inverted_dict

    return metrics_per_assay

In [ ]:
def graph_feature_set_metrics_per_assay(
    all_metrics_per_assay: Dict[str, Dict[str, Dict[str, Dict[str, Dict[str, float]]]]],
    input_sizes: Dict[str, int],
    logdir: Path | None = None,
    sort_by_input_size: bool = False,
    name: str | None = None,
    y_range: Tuple[float, float] | None = None,
    boxpoints: str = "outliers",
) -> None:
    """Graph the metrics for all feature sets, per assay, with separate plots for accuracy and F1-score.

    Args:
        all_metrics_per_assay (Dict[str, Dict[str, Dict[str, Dict[str, Dict[str, float]]]]]): A dictionary containing all metrics per assay for all feature sets.
            Format: {assay: {feature_set: {task_name: {split_name: metric_dict}}}}
        input_sizes (Dict[str, int]): A dictionary containing the input sizes for all feature sets.
        logdir (Path): The directory where the figures will be saved. If None, the figures will only be displayed.
        sort_by_input_size (bool): Whether to sort the feature sets by input size.
        name (str|None): The name of the figure.
        y_range (Tuple[float, float]|None): The y-axis range for the plots.
        boxpoints (str): The type of points to display in the box plots. Defaults to "outliers".
    """
    valid_boxpoints = ["all", "outliers"]
    if boxpoints not in valid_boxpoints:
        raise ValueError(f"Invalid boxpoints value. Choose from {valid_boxpoints}.")

    fig_assay_order = [
        "rna_seq",
        "h3k27ac",
        "h3k4me1",
        "h3k4me3",
        "h3k36me3",
        "h3k27me3",
        "h3k9me3",
        "input",
        "wgbs",
    ]

    reference_assay = next(iter(all_metrics_per_assay))
    reference_feature_set = next(iter(all_metrics_per_assay[reference_assay]))
    metadata_categories = list(
        all_metrics_per_assay[reference_assay][reference_feature_set].keys()
    )

    for _, category in enumerate(metadata_categories):
        for metric, metric_name in [
            ("Accuracy", "Accuracy"),
            ("F1_macro", "F1-score (macro)"),
        ]:
            fig = go.Figure()

            feature_sets = list(all_metrics_per_assay[reference_assay].keys())
            unique_feature_sets = set(feature_sets)
            for assay in fig_assay_order:
                if set(all_metrics_per_assay[assay].keys()) != unique_feature_sets:
                    raise ValueError("Different feature sets through assays.")

            feature_set_order = feature_sets
            if sort_by_input_size:
                feature_set_order = sorted(
                    feature_set_order, key=lambda x: input_sizes[x]
                )

            # Adjust spacing so each assay group has dedicated space based on the number of feature sets
            spacing_multiplier = (
                1.1  # Increase this multiplier if needed to add more spacing
            )
            x_positions = {
                assay: i * len(feature_set_order) * spacing_multiplier
                for i, assay in enumerate(fig_assay_order)
            }

            for i, feature_set_name in enumerate(feature_set_order):
                resolution = (
                    feature_set_name.replace("_none", "")
                    .replace("hg38_", "")
                    .split("_")[0]
                )
                color = resolution_colors[resolution]
                display_name = feature_set_name.replace("_none", "").replace("hg38_", "")

                for assay in fig_assay_order:
                    if feature_set_name not in all_metrics_per_assay[assay]:
                        continue

                    tasks_dicts = all_metrics_per_assay[assay][feature_set_name]

                    if feature_set_name not in input_sizes:
                        print(f"Skipping {feature_set_name}, no input size found.")
                        continue

                    task_name = category
                    if "split" in task_name:
                        raise ValueError("Split in task name. Wrong metrics dict.")

                    try:
                        task_dict = tasks_dicts[task_name]
                    except KeyError:
                        print(
                            f"Skipping {feature_set_name}, {task_name} for assay {assay}"
                        )
                        continue

                    y_vals = [task_dict[split][metric] for split in task_dict]
                    hovertext = [
                        f"{assay} - {display_name} - {split}: {metrics_dict[metric]:.4f}"
                        for split, metrics_dict in task_dict.items()
                    ]

                    x_position = x_positions[assay] + i
                    fig.add_trace(
                        go.Box(
                            x=[x_position] * len(y_vals),
                            y=y_vals,
                            name=f"{assay}|{display_name}",
                            boxmean=True,
                            boxpoints=boxpoints,
                            marker=dict(size=3, color="black"),
                            line=dict(width=1, color="black"),
                            fillcolor=color,
                            hovertemplate="%{text}",
                            text=hovertext,
                            showlegend=False,
                            legendgroup=display_name,
                        )
                    )

                    # separate box groups
                    fig.add_vline(
                        x=x_positions[assay] - 1, line_width=1, line_color="black"
                    )

            # Add dummy traces for the legend
            for feature_set_name in feature_set_order:
                resolution = (
                    feature_set_name.replace("_none", "")
                    .replace("hg38_", "")
                    .split("_")[0]
                )
                color = resolution_colors[resolution]
                display_name = feature_set_name.replace("_none", "").replace("hg38_", "")

                fig.add_trace(
                    go.Scatter(
                        name=display_name,
                        x=[None],
                        y=[None],
                        mode="markers",
                        marker=dict(size=10, color=color),
                        showlegend=True,
                        legendgroup=display_name,
                    )
                )

            title = f"Neural network performance - {category} - {metric_name} (per assay)"
            if name is not None:
                title += f" - {name}"
            fig.update_layout(
                width=1500,
                height=1000,
                title=title,
                xaxis_title="Assay",
                yaxis_title=metric_name,
            )

            # Create x-axis labels
            fig.update_xaxes(
                tickmode="array",
                tickvals=[
                    x_positions[assay] + len(feature_set_order) / 2
                    for assay in fig_assay_order
                ],
                ticktext=list(x_positions.keys()),
                title="Assay",
            )

            fig.update_layout(
                legend=dict(
                    title="Feature Sets", itemsizing="constant", traceorder="normal"
                )
            )
            if y_range:
                fig.update_yaxes(range=y_range)

            if logdir:
                base_name = f"feature_set_metrics_{category}_{metric}_per_assay"
                if name is not None:
                    base_name = base_name + f"_{name}"
                save_figure(fig, logdir, base_name)

            fig.show()

In [ ]:
all_results = split_results_handler.obtain_all_feature_set_data(
    parent_folder=mixed_data_dir,
    merge_assays=True,
    return_type="split_results",
    include_categories=[CELL_TYPE],
    include_sets=metric_orders_map[set_selection_name],
    exclude_names=["16ct", "27ct", "7c", "chip-seq-only"],
)

In [ ]:
metrics_per_assay = prepare_metric_sets_per_assay(all_results)  # type: ignore

In [ ]:
# Reorder feature sets
feature_set_order = metric_orders_map[set_selection_name]
for assay, feature_sets in list(metrics_per_assay.items()):
    metrics_per_assay[assay] = {
        feature_set_name: metrics_per_assay[assay][feature_set_name]
        for feature_set_name in feature_set_order
    }

In [ ]:
# preplot_path = plot_data_dir / f"fig{fig_N}_D_metrics_per_assay.json"
# with open(preplot_path, "w", encoding="utf-8") as f:
#     json.dump(metrics_per_assay, f, indent=4)

In [ ]:
graph_feature_set_metrics_per_assay(
    all_metrics_per_assay=metrics_per_assay,  # type: ignore
    input_sizes=input_sizes,
    boxpoints="all",
    sort_by_input_size=False,
    y_range=(0.295, 1.005),
)

## Panel E - SHAP values: Gene ontology enrichment analysis

### Prep data

In [ ]:
selected_cell_types = [
    "T_cell",
    "neutrophil",
    "lymphocyte_of_B_lineage",
    "brain",
]
go_terms_table = [
    "T cell receptor complex",
    "plasma membrane signaling receptor complex",
    "adaptive immune response",
    # "receptor complex",
    "secretory granule",
    "secretory vesicle",
    "secretory granule membrane",
    # "intracellular vesicle",
    "immunoglobulin complex",
    "immune response",
    # "immune system process",
    "homophilic cell adhesion via plasma membrane adhesion molecules",
    "DNA binding",
    "cell-cell adhesion via plasma-membrane adhesion molecules",
    # "RNA polymerase II cis-regulatory region sequence-specific DNA binding",
    # "blood microparticle",
    # "platelet alpha granule lumen",
    # "fibrinogen complex",
    # "endoplasmic reticulum lumen",
]

In [ ]:
SHAP_dir = base_data_dir / "SHAP"
if not SHAP_dir.exists():
    raise FileNotFoundError(f"Directory {SHAP_dir} does not exist.")

cell_type_shap_dir = (
    SHAP_dir / "hg38_100kb_all_none" / f"{CELL_TYPE}_1l_3000n" / "10fold-oversampling"
)
beds_file = cell_type_shap_dir / "select_beds_top303.tar.gz"
if not beds_file.exists():
    raise FileNotFoundError(f"File {beds_file} does not exist.")

In [ ]:
# for G
ct_important_bins: Dict[str, List[int]] = {}

all_go_dfs: Dict[str, pd.DataFrame] = {}
with tarfile.open(beds_file, "r:gz") as tar:
    for member in tar.getmembers():
        filename = member.name

        # for GO terms
        if filename.endswith("profiler.tsv") and "merge_samplings" in filename:
            with tar.extractfile(member) as f:  # type: ignore
                go_df = pd.read_csv(f, sep="\t", index_col=0)
                all_go_dfs[member.name] = go_df

        # for index of important bins
        if "merge_samplings" in filename and filename.endswith("bed"):
            file_obj: IO[bytes] = tar.extractfile(member)  # type: ignore

            cell_type = (
                filename.split("/")[1]
                .replace("merge_samplings_", "")
                .replace("_features.bed", "")
                .lower()
            )

            ct_important_bins[cell_type] = bed_to_bins(
                file_obj, chroms=chromsizes, resolution=100 * 1000
            )

assert len(all_go_dfs) == 16

In [ ]:
for name, df in all_go_dfs.items():
    sub_df = df.copy()
    sub_df.loc[:, "shap_source"] = re.match(r".*/merge_samplings_(.*)_features_intersect_gff_gprofiler.tsv", name).group(1)  # type: ignore
    sub_df.loc[:, "table_val"] = -np.log10(sub_df.loc[:, "p_value"])
    all_go_dfs[name] = sub_df

In [ ]:
full_concat_df = pd.concat(all_go_dfs.values())
full_concat_df = full_concat_df.drop(["significant", "query"], axis=1)

assert all(go_term in full_concat_df["name"].values for go_term in go_terms_table)

In [ ]:
preplot_path = plot_data_dir / f"fig{fig_N}_E_concat_gprofiler.tsv"
full_concat_df.to_csv(preplot_path, sep="\t", index=False)

In [ ]:
table = full_concat_df.pivot_table(
    index="name", columns="shap_source", values="table_val", aggfunc="mean"
)

### Graph

In [ ]:
# include line break for graph labels
go_terms_graph = [
    "T cell receptor complex",
    "plasma membrane<br>signaling receptor complex",
    "adaptive immune response",
    # "receptor complex",
    "secretory granule",
    "secretory vesicle",
    "secretory granule membrane",
    # "intracellular vesicle",
    "immunoglobulin complex",
    "immune response",
    # "immune system process",
    "homophilic cell adhesion via<br>plasma membrane adhesion molecules",
    "DNA binding",
    "cell-cell adhesion via<br>plasma-membrane adhesion molecules",
    #     "RNA polymerase II cis-regulatory<br>region sequence-specific DNA binding",
    #     "blood microparticle",
    #     "platelet alpha granule lumen",
    #     "fibrinogen complex",
    #     "endoplasmic reticulum lumen",
]

In [ ]:
# Keep only selected cell types and GO terms
sub_table = table.loc[go_terms_table, selected_cell_types].copy()
assert sub_table.shape == (len(go_terms_graph), len(selected_cell_types))

In [ ]:
# Rename index
sub_table = sub_table.rename(index=dict(zip(go_terms_table, go_terms_graph)))

In [ ]:
sigma = "\u03c3"

colorbar = dict(
    title="-log<sub>10</sub>(p-value)",
    tickvals=[0, 1.30, 2, 3, 5, 6.53, 10],
    ticktext=[
        "0",
        f"1.30: p=0.05~2{sigma}",
        "2: p=0.01",
        f"3: p=0.001~3{sigma}",
        "5",
        f"6.53: p=3x10<sup>-7</sup> = 5{sigma}",
        "10",
    ],
)

In [ ]:
# Fill z with NaNs (for color) and make custom text labels
z = sub_table.values  # keep NaNs in for color
text = np.where(
    np.isnan(z), "NS", np.char.mod("%.2f", z)  # format floats to 2 decimal places
)

fig = go.Figure(
    data=go.Heatmap(
        z=np.nan_to_num(
            z, nan=0
        ),  # replace NaNs with 0 for coloring (or use a custom colormap later)
        x=sub_table.columns,
        y=sub_table.index,
        colorscale="Blues",
        zmin=0,
        zmax=10,
        colorbar=colorbar,
        text=text,
        texttemplate="%{text}",
        hovertemplate="GO Term: %{y}<br>Class: %{x}<br>Value: %{z:.2f}<extra></extra>",
        showscale=True,
        xgap=2,
        ygap=2,
    )
)

# Customize layout
fig.update_layout(
    width=600,
    height=800,
    plot_bgcolor="black",
    margin=dict(t=120),
    title={
        "text": "Top SHAP regions: GO term enrichment",
        "y": 0.99,  # Position closer to top
        "x": 0.01,
        "xanchor": "left",
        "yanchor": "top",
    },
)
# Fix gridlines
fig.update_xaxes(showgrid=False, side="top")
fig.update_yaxes(showgrid=False, autorange="reversed")

fig.update_layout()

fig.show()

## Panel G - Max Chromscore for important SHAP features for certain cell types

Same cell types as in panel E + hepatocyte

### Important SHAP regions prep

Using the bin index (100kb bins) for the important SHAP regions computed in section E, create a cell type / bin index mapping

In [ ]:
all_bins = set()
all_bins_list = []
for bins in ct_important_bins.values():
    all_bins.update(bins)
    all_bins_list.extend(bins)

all_bins = sorted(all_bins)
print(len(all_bins), "unique important bins across cell types.")
print(len(all_bins_list), "total important bins across cell types.")

In [ ]:
# Find relevant cell types for each bin, optimized for pandas future vectorization
relevant_pairs_list = []
for cell_type, bins_list in ct_important_bins.items():
    for bin_idx in bins_list:
        relevant_pairs_list.append({"bin_index": bin_idx, CELL_TYPE: cell_type})

bin_to_relevant_ct_df = pd.DataFrame(relevant_pairs_list)
bin_to_relevant_ct_df["bin_index"] = bin_to_relevant_ct_df["bin_index"].astype(int)

assert bin_to_relevant_ct_df.shape[0] > len(ct_important_bins)

In [ ]:
print(bin_to_relevant_ct_df.shape)
print(bin_to_relevant_ct_df["bin_index"].nunique())
print("\nImportant regions for each cell type:")
print(bin_to_relevant_ct_df[CELL_TYPE].value_counts(dropna=False))

In [ ]:
classifier_cell_types = set(bin_to_relevant_ct_df[CELL_TYPE].unique())
assert len(classifier_cell_types) == 16

### ChromScore prep

Maximum ChromScore values computed directly from bigwigs, for each 100kb regions.

In [ ]:
chromscore_dir = paper_dir / "data" / "ChromScore"
chromscore_file = chromscore_dir / "max_metrics_clean.h5"
if not chromscore_file.exists():
    raise FileNotFoundError(f"{chromscore_file} does not exist")

print(f"Loading {chromscore_file}")
chromscores_df: pd.DataFrame = pd.read_hdf(chromscore_file)  # type: ignore
N_files, N_bins = chromscores_df.shape
print(f"Chromscores: {N_files} files, {N_bins} 100kb bins")

display(chromscores_df.head(n=2))

In [ ]:
chromscores_df["epirr"] = chromscores_df.index.str.split(".").str[0]

# Create a mapping from epirr_id_without_version to cell_type
epirr_to_cell_type = dict(
    metadata_df.loc[:, ["epirr_id_without_version", CELL_TYPE]].values
)

chromscores_df[CELL_TYPE] = (
    chromscores_df["epirr"].map(epirr_to_cell_type).str.replace(" ", "_").str.lower()
)
display(chromscores_df[CELL_TYPE].value_counts(dropna=False))

In [ ]:
# Only keep files from classifier 16ct
condition = chromscores_df[CELL_TYPE].isin(classifier_cell_types)
print(
    f"Keeping {condition.sum()} files out of {len(chromscores_df)} from classifier cell types."
)
chromscores_df = chromscores_df[condition]
assert chromscores_df[CELL_TYPE].nunique() == 16

#### Find max chromScore for each bin (for their associated files through SHAP analysis)

In [ ]:
# Melting global chromscores for future operation, now all values are in one column
# Assuming all columns starting with chr are bins, and all 100kb bins are present
region_cols_mapper = {
    col: idx
    for idx, col in enumerate(chromscores_df.columns)
    if isinstance(col, str) and col.startswith("chr")
}
assert len(region_cols_mapper) == 30321

In [ ]:
chromscores_df.rename(columns=region_cols_mapper, inplace=True)  # type: ignore

In [ ]:
melted_chromscores = chromscores_df.reset_index().rename(columns={"index": "filename"})
melted_chromscores = melted_chromscores.melt(
    id_vars=["epirr", CELL_TYPE],
    value_vars=list(region_cols_mapper.values()),
    var_name="bin_index",
    value_name="chromscore_value",
)
melted_chromscores["bin_index"] = melted_chromscores["bin_index"].astype(int)
print(melted_chromscores.shape)
print(melted_chromscores["bin_index"].nunique())
print(melted_chromscores["epirr"].nunique())
display(melted_chromscores.head(n=2))

In [ ]:
# Merge melted chromscores with bin_to_relevant_ct_df, efficient for pandas
# Filters out all irrelevant bins
merged_df = pd.merge(
    melted_chromscores, bin_to_relevant_ct_df, on=["bin_index", CELL_TYPE], how="inner"
)
print(merged_df.shape)
print(merged_df["bin_index"].nunique())
print(merged_df["epirr"].nunique())
display(merged_df.head(n=2))

In [ ]:
# Keep only columns of interest for plotting
merged_df = merged_df[["epirr", CELL_TYPE, "bin_index", "chromscore_value"]]

In [ ]:
# Sanity check, missing values
groupby = merged_df.groupby(CELL_TYPE)["chromscore_value"].apply(lambda x: x.isna().sum())
if not groupby.sum() == 0:
    display(groupby)
    raise ValueError("Missing values in chromscore_value")

assert melted_chromscores["bin_index"].nunique() == 30321

In [ ]:
# preplot_path = (
#     plot_data_dir / f"fig{fig_N}_G_max_chromscore_important_bins_per_cell_type.tsv.gz"
# )
# merged_df.to_csv(preplot_path, sep="\t", index=False, header=True, compression="gzip")

In [ ]:
# preplot_path = plot_data_dir / f"fig{fig_N}_G_max_chromscore_all_files_ID.tsv.gz"
# melted_chromscores[["epirr", CELL_TYPE]].to_csv(
#     preplot_path, sep="\t", index=False, header=True, compression="gzip"
# )

In [ ]:
# preplot_path = plot_data_dir / f"fig{fig_N}_G_max_chromscore_all_files_values.npz"
# np.savez_compressed(
#     preplot_path,
#     bin_indices=np.array(
#         melted_chromscores["bin_index"].astype(int).values, dtype=np.int32
#     ),
#     chromscore_values=np.array(
#         melted_chromscores["chromscore_value"].astype(float).values, dtype=np.float32
#     ),
#     allow_pickle=False,
# )

### Plot results, global max chromscore vs per cell type, for important SHAP regions only

In [ ]:
def prepare_chromscore_per_biospecimen_data(
    selected_bins_df: pd.DataFrame,
    all_chromscores_df: pd.DataFrame,
    cell_types_col: str = CELL_TYPE,
    allowed_cell_types: List[str] | None = None,
) -> Dict[str, Dict[str, List[float] | int]]:
    """
    Prepare plot data for chromscore per biospecimen plot.

    Each biospecimen need values for:
    - important features
    - global distribution

    This is done with independent file subsets.
    """
    grouped_means = {}

    for biospecimen, df in selected_bins_df.groupby(by=cell_types_col):
        if (allowed_cell_types is not None) and (biospecimen not in allowed_cell_types):
            print(f"Skipping {biospecimen}.")
            continue
        print(f"Processing {biospecimen}")

        all_chromscores_group = all_chromscores_df.loc[
            all_chromscores_df[cell_types_col] == biospecimen, :
        ]
        if not all_chromscores_group["chromscore_value"].isna().sum() == 0:
            raise ValueError(f"Missing values in chromscore_value for {biospecimen}")
        nb_files = df["epirr"].nunique()

        # Important features
        avg_per_bin = df.groupby("bin_index")["chromscore_value"].mean().to_list()

        # Global distribution
        all_means_file_subset = (
            all_chromscores_group.groupby("bin_index")["chromscore_value"]
            .mean()
            .to_list()
        )

        grouped_means[biospecimen] = {
            "avg_per_bin": avg_per_bin,
            "all_means_file_subset": all_means_file_subset,
            "nb_files": nb_files,
        }

    grouped_means["all_files"] = (
        all_chromscores_df.groupby("bin_index")["chromscore_value"].mean().to_list()
    )

    return grouped_means

In [ ]:
def cohens_d(x, y):
    """
    Calculate Cohen's d for independent samples.
    Uses pooled standard deviation accounting for unequal variances.
    """
    n1, n2 = len(x), len(y)
    var1, var2 = np.var(x, ddof=1), np.var(y, ddof=1)

    # Pooled standard deviation
    pooled_std = np.sqrt(((n1 - 1) * var1 + (n2 - 1) * var2) / (n1 + n2 - 2))

    # Cohen's d
    d = (np.mean(x) - np.mean(y)) / pooled_std

    return d


def test_distribution(
    x: List[float], y: List[float], verbose: bool = True
) -> Tuple[Any, Any, float]:
    """Test for distribution difference. x is used as reference for the number of samples.

    Welch's t-test and Brunner-Munzel test are computed.

    Returns:
        Tuple[TtestResult, BrunnerMunzelResult, float]: test results and cohen's d effect size.
    """
    if verbose:
        print(f"Number of samples in x: {len(x)}")
        print(f"Number of samples in y: {len(y)}")

    Welch_test = stats.ttest_ind(
        a=x,
        b=y,
        equal_var=False,
        alternative="two-sided",
        nan_policy="raise",
    )

    BM_test = stats.brunnermunzel(
        x,
        y,
        alternative="two-sided",
        nan_policy="raise",
        distribution="t",
    )

    effect_size = cohens_d(x, y)
    if verbose:
        print(f"Cohen's d effect size: {effect_size:.4f}")
        print(
            f"Welch's t-test: statistic={Welch_test.statistic:.4f}, pvalue={Welch_test.pvalue:.4e}"
        )
        print(
            f"Brunner-Munzel test: statistic={BM_test.statistic:.4f}, pvalue={BM_test.pvalue:.4e}"
        )

    return Welch_test, BM_test, effect_size


def define_pval_label(pval: float) -> str:
    """Define p-value label."""
    pval_symbol = ""
    if pval < 0.001:
        pval_symbol = "<0.001***"
    elif pval < 0.01:
        pval_symbol = "<0.01**"
    elif pval < 0.05:
        pval_symbol = "<0.05*"
    elif pval >= 0.05:
        pval_symbol = ">0.05 NS"

    return pval_symbol

In [ ]:
def plot_chromscore_per_biospecimen_violin(
    graph_data: Dict[str, Dict[str, List[float] | int]],
    cell_types: List[str] | None = None,
    logdir: Path | None = None,
    do_subplots: bool = True,
    filename: str = "important_features_16ct_max_chromscore_100kb_per_biospecimen_2violin",
    verbose: bool = False,
) -> pd.DataFrame:
    """
    Plot average of 'max chromscore' per biospecimen as violin plots,
    using regions and files per biospecimen independently.

    The average is computed over files, so one averaged value per feature/bin/region.

    Args:
        graph_data: Dict[str, Dict[str, List[float] | int]]. From prepare_chromscore_per_biospecimen_data.
        cell_types: List[str]|None. List of cell types to plot.

    Returns:
        pd.DataFrame. Dataframe with scipy.stats results objects.
    """
    data = copy.deepcopy(graph_data)
    if not cell_types:
        cell_types = list(data.keys())
        cell_types.remove("all_files")

    colors = px.colors.qualitative.Dark24[0:2]

    fig = go.Figure()
    if do_subplots:
        fig = make_subplots(
            rows=4,
            cols=4,
            shared_yaxes=True,
            vertical_spacing=0.075,
            horizontal_spacing=0.025,
            y_title="Average of max value in selected regions of 100kb (over files)",
        )

    # Filter
    try:
        data = {biospecimen: graph_data[biospecimen] for biospecimen in cell_types}
    except KeyError as err:
        raise KeyError(
            f"A cell type is missing from the graph_data.\ncell types: {graph_data.keys()}.\nDesired: {cell_types}."
        ) from err

    all_tests = []
    trace_names = []
    for idx, (biospecimen, data) in enumerate(data.items()):
        if biospecimen not in cell_types:
            continue

        avg_per_bin: List[float] = data["avg_per_bin"]  # type: ignore
        all_means_file_subset: List[float] = data["all_means_file_subset"]  # type: ignore

        nb_files = data["nb_files"]
        nb_features = len(avg_per_bin)

        if do_subplots:
            placement_dict = {
                "row": idx // 4 + 1,
                "col": idx % 4 + 1,
            }
        else:
            placement_dict = {}

        # Important features
        show_points = False
        if len(avg_per_bin) <= 10:
            show_points = "all"
        fig.add_trace(
            go.Violin(
                side="negative",
                name=f"trace{idx}",
                y=avg_per_bin,
                fillcolor=colors[0],
                line=dict(color="black", width=1.5 if do_subplots else 0),
                marker_size=3,
                jitter=0.1,
                pointpos=-0.4,
                showlegend=False,
                meanline_visible=True,
                points=show_points,
                spanmode="hard",
                legendgroup="All features",
                box=dict(
                    visible=True,
                    fillcolor=colors[0] if do_subplots else "black",
                    width=0.4,
                    line_width=0.5 if do_subplots else 0,
                ),
                scalemode="width",  # occupy all possible space for subplots
                scalegroup=f"trace{idx}",
            ),
            **placement_dict,  # type: ignore
        )

        # Global distribution comparison
        fig.add_trace(
            go.Violin(
                side="positive",
                name=f"trace{idx}",
                y=all_means_file_subset,
                fillcolor=colors[1],
                line=dict(color="black", width=1.5 if do_subplots else 0),
                showlegend=False,
                meanline_visible=True,
                points=False,
                spanmode="hard",
                legendgroup="All features",
                box=dict(
                    visible=True,
                    fillcolor=colors[1] if do_subplots else "black",
                    width=0.4,
                    line_width=0.5 if do_subplots else 0,
                ),
                scalemode="width",
                scalegroup=f"trace{idx}",
            ),
            **placement_dict,  # type: ignore
        )

        welchtest, bmtest, cohen_d = test_distribution(
            x=avg_per_bin,
            y=all_means_file_subset,
            verbose=False,
        )
        pvals = [welchtest.pvalue, bmtest.pvalue]
        if verbose:
            print(f"{biospecimen}, {nb_features} features, {nb_files} files")
            print(f"pvals [Welch, BM]: {pvals}\n\n")

        all_tests.append(
            [
                biospecimen,
                nb_files,
                nb_features,
                len(all_means_file_subset),
                welchtest,
                bmtest,
                cohen_d,
            ]
        )

        pval = float(np.max(pvals))
        pval_symbol = define_pval_label(pval * 16)

        if do_subplots:
            group_name = f"{biospecimen}<br>({nb_files} files, {nb_features} features)<br>p{pval_symbol}"
            fig.update_xaxes(
                showticklabels=False,
                row=idx // 4 + 1,
                col=idx % 4 + 1,
                title=group_name,
                title_standoff=2,
                title_font=dict(size=10),
            )
        else:
            group_name = f"{biospecimen} ({nb_files} files, {nb_features} features), p{pval_symbol}"
            trace_names.append(group_name)

    # Manually set names for traces
    if not do_subplots:
        newnames = {f"trace{idx}": name for idx, name in enumerate(trace_names)}
        fig.for_each_trace(lambda t: t.update(name=newnames[t.name]))

    # Legend with dummy points
    for i, name in enumerate(["Important SHAP features", "All features"]):
        fig.add_trace(
            go.Scatter(
                x=[None],
                y=[None],
                mode="markers",
                name=name,
                legendgroup=name,
                showlegend=True,
                marker=dict(color=colors[i], symbol="square"),
            ),
        )

    fig.update_yaxes(range=[0, 1])

    fig.update_layout(
        title="ChromScore per biospecimen file subset",
        width=1000,
        height=900,
        legend=dict(
            itemsizing="constant",
            yanchor="top",
            xanchor="right",
            y=1.1,
            x=0.8,
        ),
    )

    if not do_subplots:
        fig.update_layout(
            yaxis_title="Average of max value in selected regions of 100kb (over files)",
            xaxis_title="Biospecimen",
            width=700,
            height=700,
        )

    fig.show()

    if logdir is not None:
        print("Saving figure.")
        save_figure(fig, logdir, filename, scale=1.5)

    return pd.DataFrame(
        all_tests,
        columns=[
            "biospecimen",
            "nb_files",
            "Nb features (N_1)",
            "Nb features global (N_2)",
            "test_Welch",
            "test_BM",
            "cohen_d_effect_size",
        ],
    )

In [ ]:
cell_types = ["t_cell", "neutrophil", "lymphocyte_of_b_lineage", "brain", "hepatocyte"]

In [ ]:
graph_data = prepare_chromscore_per_biospecimen_data(
    selected_bins_df=merged_df,
    all_chromscores_df=melted_chromscores,
    allowed_cell_types=cell_types,
)

In [ ]:
# preplot_path = (
#     plot_data_dir / f"fig{fig_N}_G_max_chromscore_important_bins_per_cell_type.json"
# )
# with open(preplot_path, "w", encoding="utf-8") as f:
#     json.dump(graph_data, f, indent=4)

In [ ]:
stats_df = plot_chromscore_per_biospecimen_violin(
    graph_data=graph_data,
    do_subplots=False,
    cell_types=cell_types,
)

Notes:
- The above p-values of the graph are corrected for multiple hypothesis testing using a Bonferroni correction for 16 tests (multiplied by 16). See chromscore.ipynb for full details.

- The max chromscore values were computed directly from bigwig files using pyBigWig, for each 100kb bin, and then, for each bin/region, those max values were averaged across the relevant cell type files (e.g. for T cells, across all T cell files). The number of point per violin corresponds to the number of important SHAP regions/features for that cell type. This is compared to the average max chromscore across the same files, for all 30321 100kb bins. This means the left violing is a subset of the right violin.

## Panel H - Inference on other public sources

Namely, ChIP-Atlas and recount3.

In [ ]:
metrics_dir = paper_dir / "tables" / "dfreeze_v2" / "predictions" / "metrics"
if not metrics_dir.exists():
    raise FileNotFoundError(f"Directory {metrics_dir} does not exist.")

In [ ]:
chip_atlas_path = metrics_dir / "C-A_metrics_per_assay.tsv"
recount3_path = metrics_dir / "recount3_metrics_per_assay_assay11c-filtered.tsv"

chip_atlas_df = pd.read_csv(chip_atlas_path, sep="\t")
recount3_df = pd.read_csv(recount3_path, sep="\t")

In [ ]:
chip_atlas_df["source_file"] = chip_atlas_path.name
recount3_df["source_file"] = recount3_path.name

summary_df = pd.concat([chip_atlas_df, recount3_df], ignore_index=True)

In [ ]:
# remap categories so they have the same task_names
summary_df["task_name"] = summary_df["task_name"].replace(
    {
        "harmonized_biomaterial_type": "biomaterial_type",
        "biomat": "biomaterial_type",
        "harmonized_donor_sex": "sex",
        "harmonized_sample_cancer_high": "cancer_status",
        "cancer": "cancer_status",
    }
)

In [ ]:
preplot_path = plot_data_dir / f"fig{fig_N}_H_metrics_per_assay_public_DB.tsv"
summary_df.to_csv(preplot_path, sep="\t", index=False, na_rep="")

In [ ]:
# we want extract low/high confidence pred per DB (>=0.6), and missing labels
assay_epiclass_labels = ["avg-all", "count-unknown"]
summary_df = summary_df[summary_df["assay_epiclass"].isin(assay_epiclass_labels)]

In [ ]:
# remap categories so they have the same task_names
summary_df["task_name"] = summary_df["task_name"].replace(
    {
        "harmonized_biomaterial_type": "biomaterial_type",
        "biomat": "biomaterial_type",
        "harmonized_donor_sex": "sex",
        "harmonized_sample_cancer_high": "cancer_status",
        "cancer": "cancer_status",
    }
)

In [ ]:
categories_to_plot = ["cancer_status", "sex", "biomaterial_type"]

# Formatting data for plotting
graph_values = []
for category in categories_to_plot:
    df = summary_df[summary_df["task_name"] == category]

    # Known labels
    # nb known labels is going to be number of samples for avg-all min_predScore => 0.0
    N_known_labels = df[
        (df["assay_epiclass"] == "avg-all") & (df["min_predScore"].astype(str) == "0.0")
    ]["nb_samples"].sum()
    graph_values.append([category, N_known_labels, "Provided/Extracted"])

    # Unknown labels
    unknown_df_cond = df["assay_epiclass"] == "count-unknown"
    all_pred_cond = df["min_predScore"].astype(str) == "0.0"
    high_pred_cond = df["min_predScore"].astype(str) == "0.6"

    N_total_unknown = df[unknown_df_cond & all_pred_cond]["nb_samples"].sum()
    N_high_conf = df[unknown_df_cond & high_pred_cond]["nb_samples"].sum()

    N_low_conf = N_total_unknown - N_high_conf

    graph_values.append([category, N_high_conf, "High Conf Pred"])
    graph_values.append([category, N_low_conf, "Low Conf Pred"])

plot_df = pd.DataFrame(data=graph_values, columns=["Category", "Count", "Type"])

In [ ]:
# Define colors for graphs
color_map = {
    "Low Conf Pred": "#7fccd6",  # light blue
    "High Conf Pred": "#004e58",  # dark blue
    "Provided/Extracted": "#000000",  # black
}

In [ ]:
# Plot percentage of samples per category and type
fig_pct = px.bar(
    plot_df,
    x="Category",
    y="Count",
    color="Type",
    color_discrete_map=color_map,
    title="Data Availability and Prediction Confidence",
    labels={"Count": "Percentage of public data samples (%)"},
    category_orders={"Type": ["Provided/Extracted", "High Conf Pred", "Low Conf Pred"]},
)

# Update layout to normalize bars to 100%
fig_pct.update_layout(
    barnorm="percent",
    height=600,
    width=700,
    legend_traceorder="reversed",
)

fig_pct.update_yaxes(ticksuffix="%", range=[0, 101])

fig_pct.show()

In [ ]:
# Plot number of samples per category and type
fig = px.bar(
    plot_df,
    x="Category",
    y="Count",
    color="Type",
    color_discrete_map=color_map,
    title="Data Availability and Prediction Confidence",
    labels={"Count": "Number of Samples"},
    category_orders={"Type": ["Provided/Extracted", "High Conf Pred", "Low Conf Pred"]},
)

fig.update_layout(
    barnorm=None,
    height=600,
    width=700,
    legend_traceorder="reversed",
)

fig.show()